# ST456 Colab 运行器


In [ ]:
# 运行 notebook 前先检查这些设置
ZIP_EXPECTED_HINT = 'ST456group.zip'
PROJECT_SUBDIR = 'text'

USE_GOOGLE_DRIVE = False
GOOGLE_DRIVE_WORKSPACE = '/content/drive/MyDrive/st456_runs'

# False 表示进行完整训练和完整测试集评估。
QUICK_VALIDATION = False

# 一次性训练 E1-E5。
RUN_FULL_MAINLINE = True

# 同时运行 retrieval 附录实验。
RUN_APPENDIX_RETRIEVAL = True

# 下载一个包含指标和运行配置的 ZIP 文件。
DOWNLOAD_RESULTS_ZIP = True

print('设置已加载。这个 notebook 接下来会让你上传 ZIP 文件。')


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile

# 避免模型下载进度条刷屏。
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'

def run(cmd, cwd=None):
    """运行一条命令，并在屏幕上保留一行实时状态。"""
    import html
    from collections import deque
    from IPython.display import HTML, display

    print(f'\n>>> {cmd}', flush=True)
    status_handle = display(
        HTML(f"<pre>>>> {html.escape(cmd)}\n{html.escape('开始运行...')}</pre>"),
        display_id=True,
    )
    tail = deque(maxlen=80)

    def update_status(text):
        safe_cmd = html.escape(cmd)
        safe_text = html.escape(text or '正在运行...')
        status_handle.update(HTML(f"<pre>>>> {safe_cmd}\n{safe_text}</pre>"))

    process = subprocess.Popen(
        cmd, shell=True, cwd=cwd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )

    current = ''
    while True:
        char = process.stdout.read(1)
        if char == '' and process.poll() is not None:
            break
        if not char:
            continue
        if char == '\r':
            line = current.strip()
            if line:
                update_status(line[-400:])
            current = ''
            continue
        if char == '\n':
            line = current.strip()
            if line:
                tail.append(line)
                update_status(line[-400:])
            current = ''
            continue
        current += char

    final_line = current.strip()
    if final_line:
        tail.append(final_line)
        update_status(final_line[-400:])

    process.wait()
    if process.returncode != 0:
        recent_output = '\n'.join(tail) or '没有捕获到输出。'
        update_status(f'失败（退出码 {process.returncode}）')
        raise RuntimeError(
            f'命令失败（退出码 {process.returncode}）：{cmd}\n'
            f'最近输出：\n{recent_output}'
        )

    update_status('完成')

def clear_gpu_memory(stage=''):
    import gc

    gc.collect()
    try:
        import torch
    except ImportError:
        print(f'[GPU cleanup] {stage or "当前阶段"}: 未安装 torch，因此跳过这一步。')
        return

    if not torch.cuda.is_available():
        print(f'[GPU cleanup] {stage or "当前阶段"}: CUDA 不可用，因此跳过这一步。')
        return

    torch.cuda.empty_cache()
    if hasattr(torch.cuda, 'ipc_collect'):
        torch.cuda.ipc_collect()
    reserved_mb = torch.cuda.memory_reserved() / (1024 ** 2)
    allocated_mb = torch.cuda.memory_allocated() / (1024 ** 2)
    print(
        f'[GPU cleanup] {stage or "当前阶段"}: '
        f'allocated={allocated_mb:.1f} MB, reserved={reserved_mb:.1f} MB'
    )

workspace_root = Path('/content')
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    workspace_root = Path(GOOGLE_DRIVE_WORKSPACE)

workspace_root.mkdir(parents=True, exist_ok=True)
os.chdir(workspace_root)
print('当前工作目录:', Path.cwd())

from google.colab import files
print(f'请上传项目 ZIP 文件，例如: {ZIP_EXPECTED_HINT}')
uploaded = files.upload()
zip_candidates = [name for name in uploaded.keys() if name.lower().endswith('.zip')]
if not zip_candidates:
    raise ValueError('没有上传 ZIP 文件。')

zip_name = zip_candidates[0]
print('已上传 ZIP:', zip_name)
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall(workspace_root)

project_matches = sorted(workspace_root.rglob(PROJECT_SUBDIR))
project_matches = [path for path in project_matches if path.is_dir()]
if not project_matches:
    raise FileNotFoundError(f'解压后找不到 {PROJECT_SUBDIR} 目录。')

project_root = project_matches[0]
os.chdir(project_root)
src_root = project_root / 'src'
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))
print('项目目录:', Path.cwd())
print('已加入 Python 路径:', src_root)
run(f'{sys.executable} -m pip install -r requirements.txt')
print('环境配置完成。')


## 构建数据并检查 token 预算

这个 cell 会下载 Holmes 文本，构建数据划分，并为 E1、E3 和 E5 写出 token 预算摘要。


In [ ]:
run(f'{sys.executable} scripts/download_gutenberg.py')
run(f'{sys.executable} scripts/build_dataset.py --context-size 4 --min-chars 40')

Path('artifacts/eval').mkdir(parents=True, exist_ok=True)
token_stat_configs = [
    ('E1', 'configs/e1_distilgpt2_plain_full.yaml'),
    ('E3', 'configs/e3_distilgpt2_structured_long_context.yaml'),
    ('E5', 'configs/e5_distilgpt2_structured_aux_ranking.yaml'),
]

for experiment_id, config_path in token_stat_configs:
    output_path = Path('artifacts/eval') / f'token_stats_{experiment_id.lower()}.json'
    run(f'{sys.executable} scripts/inspect_token_stats.py --config {config_path} --output-path {output_path}')

print('Token 统计文件:')
for path in sorted(Path('artifacts/eval').glob('token_stats_*.json')):
    print('-', path)


## 训练主线实验

E1 总是会运行。如果 `RUN_FULL_MAINLINE = True`，notebook 会继续运行 E2-E5W。


In [ ]:
from novel_continuation.training import load_training_config

main_experiments = [
    ('E1', 'configs/e1_distilgpt2_plain_full.yaml', 'artifacts/e1_plain_full'),
    ('E2', 'configs/e2_distilgpt2_structured_full.yaml', 'artifacts/e2_structured_full'),
    ('E3', 'configs/e3_distilgpt2_structured_long_context.yaml', 'artifacts/e3_long_context'),
    ('E4', 'configs/e4_distilgpt2_structured_lora.yaml', 'artifacts/e4_lora'),
    ('E5', 'configs/e5_distilgpt2_structured_aux_ranking.yaml', 'artifacts/e5_aux_ranking'),
    ('E5W', 'configs/e5_distilgpt2_structured_aux_ranking_wide.yaml', 'artifacts/e5_aux_ranking_wide'),
]

for experiment_id, config_path, _output_dir in main_experiments:
    print(f'----- {experiment_id}: 开始训练 -----')
    run(f'{sys.executable} scripts/train_experiment.py --config {config_path} --seed 42')
    clear_gpu_memory(f'{experiment_id} 训练完成')

print('主线训练完成。')
for experiment_id, config_path, _output_dir in main_experiments:
    resolved = load_training_config(Path(config_path))
    print(f'- {experiment_id}: output_dir={resolved["output_dir"]}')


## 运行 3 个 seed 的生成和评估

这个 cell 会写出报告使用的文件：

- `generated_samples_*_seed*.jsonl`
- `metrics_*_seed*.csv`
- `metrics_*_summary.csv`
- 如果你决定导出可选附录文件，则还会有 `human_eval_*.csv`


In [ ]:
import json
eval_output_dir = Path('artifacts/eval')
eval_output_dir.mkdir(parents=True, exist_ok=True)

for experiment_id, _config_path, model_dir in main_experiments:
    print(f'\n----- {experiment_id}: 3-seed 自动评估 -----')
    run(
        f'{sys.executable} scripts/run_eval_3seed.py ' \
        f'--experiment-id {experiment_id.lower()} ' \
        f'--model-dir {model_dir} ' \
        f'--output-dir {eval_output_dir}'
    )
    clear_gpu_memory(f'{experiment_id} 评估完成')

print('\n3-seed 评估文件已准备好。')
for path in sorted(eval_output_dir.glob('*')):
    print('-', path)

print('\n比较 E5 和 E5W 的 validation_main_loss:')
for experiment_id, _config_path, model_dir in main_experiments:
    if experiment_id not in {'E5', 'E5W'}:
        continue
    training_config = Path(model_dir) / 'training_config.json'
    if not training_config.exists():
        print(f'- {experiment_id}: 找不到 {training_config}')
        continue
    payload = json.loads(training_config.read_text(encoding='utf-8'))
    validation = payload.get('metadata', {}).get('validation', {})
    print(f'- {experiment_id}: validation_main_loss={validation.get("validation_main_loss")}, validation_perplexity={validation.get("validation_perplexity")}')


## 附录 retrieval 实验

这个代码块只会在 `RUN_APPENDIX_RETRIEVAL = True` 时运行。


In [ ]:
if RUN_APPENDIX_RETRIEVAL:
    run(f'{sys.executable} scripts/train_retrieval_model.py --config configs/retrieval_distilgpt2.yaml')
    run(f'{sys.executable} scripts/generate_samples.py --model-dir artifacts/retrieval --use-retrieval --history-path data/processed/train.jsonl --output-path artifacts/eval/generated_samples_retrieval.jsonl')
    run(f'{sys.executable} scripts/run_auto_eval.py --model-dir artifacts/retrieval --generated-path artifacts/eval/generated_samples_retrieval.jsonl --output-path artifacts/eval/metrics_retrieval.csv')
    # 把 human-eval 导出保留为附录步骤，不放进默认运行流程。
    print('retrieval 附录实验已完成。')
else:
    print('notebook 已跳过 retrieval 附录实验。')


## 打包结果

当 `DOWNLOAD_RESULTS_ZIP = True` 时，这个 cell 会：

1. 删除中间 checkpoint 来节省空间
2. 打包评估输出和训练配置，但不包含模型权重


In [ ]:
import glob

if DOWNLOAD_RESULTS_ZIP:
    # 删除中间 checkpoint，避免 Colab 磁盘空间不足。
    for ckpt_dir in sorted(Path('artifacts').rglob('checkpoint-*')):
        if ckpt_dir.is_dir():
            shutil.rmtree(ckpt_dir)
            print(f'已删除: {ckpt_dir}')

    # 暂存一次运行结束后通常需要保留的文件。
    download_dir = Path('artifacts/download_package')
    download_dir.mkdir(parents=True, exist_ok=True)

    # 复制指标、生成结果、可选 human-eval CSV 和 token 统计。
    eval_src = Path('artifacts/eval')
    if eval_src.exists():
        shutil.copytree(eval_src, download_dir / 'eval', dirs_exist_ok=True)

    # 保留每个 training_config.json，用于报告表格和 sanity check。
    for config_file in Path('artifacts').glob('*/training_config.json'):
        dest = download_dir / config_file.parent.name / config_file.name
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(config_file, dest)

    # 构建 ZIP 并下载。
    archive_path = shutil.make_archive(
        str(Path('artifacts') / 'colab_results'), 'zip',
        root_dir=str(download_dir)
    )
    size_mb = Path(archive_path).stat().st_size / (1024 * 1024)
    print(f'ZIP 文件已创建: {archive_path} ({size_mb:.1f} MB)')

    # 清理临时暂存目录。
    shutil.rmtree(download_dir)

    from google.colab import files
    files.download(archive_path)
else:
    print('notebook 已跳过结果下载步骤。')
